In [5]:
import torch
import torch.nn as nn
import os, time, copy
import numpy as np
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import googlenet
from sklearn.model_selection import train_test_split


# Settings

torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cpu")   # AdaRound + eval done on CPU
NUM_CLASSES = 200
DATA_DIR = "DIR/horse/sapi279d-test_workspace/train"
WEIGHTS_PATH = "googlenet_weights.pth"


#  Dataset & DataLoaders (same as baseline)

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")

# stratified split
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    val_ids, test_ids   = train_test_split(temp_ids, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("DataLoaders ready")


# Load trained FP32 GoogLeNet

state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")

# clean keys (remove "_orig_mod.")
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

model_fp32 = googlenet(weights=None, aux_logits=True)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, NUM_CLASSES)
model_fp32.aux1.fc2 = nn.Linear(model_fp32.aux1.fc2.in_features, NUM_CLASSES)
model_fp32.aux2.fc2 = nn.Linear(model_fp32.aux2.fc2.in_features, NUM_CLASSES)

missing, unexpected = model_fp32.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
model_fp32.eval()
print("FP32 GoogLeNet loaded")


# AdaRound Helper

def adaround_weight_tensor(weight, n_bits=8, num_iters=500, lr=1e-2):
    w = weight.detach().clone()
    scale = (w.max() - w.min()) / (2 ** n_bits - 1)
    zero_point = torch.round(-w.min() / scale)

    alpha = torch.zeros_like(w, requires_grad=True)
    optimizer = torch.optim.Adam([alpha], lr=lr)

    for i in range(num_iters):
        optimizer.zero_grad()
        soft_offset = torch.sigmoid(alpha)
        w_q = torch.floor(w / scale) + soft_offset
        soft_weight = w_q * scale
        loss = torch.nn.functional.mse_loss(soft_weight, w)
        loss.backward()
        optimizer.step()
        alpha = alpha.detach().requires_grad_()
    with torch.no_grad():
        hard_offset = (torch.sigmoid(alpha) > 0.5).float()
        w_q = torch.floor(w / scale) + hard_offset
        quantized_weight = w_q * scale
    return quantized_weight

def apply_adaround(model, n_bits=8, num_iters=500, lr=1e-2):
    new_model = copy.deepcopy(model)   # safe copy
    for name, param in new_model.named_parameters():
        if "weight" in name and param.dim() > 1:  # conv/linear weights
            print(f"AdaRounding: {name}")
            param.data = adaround_weight_tensor(param.data, n_bits, num_iters, lr)
    return new_model


# Apply AdaRound

model_adaround = apply_adaround(model_fp32, n_bits=8, num_iters=500, lr=1e-2)


# Evaluation

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            if isinstance(outputs, tuple):  # GoogLeNet returns (main, aux1, aux2)
                outputs = outputs[0]
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_ada  = evaluate_model(model_adaround, test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"AdaRound INT8 Accuracy: {acc_ada:.2f}%")


# Size + Runtime

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=20):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    return (total_time / num_batches) * 1000, (total_time / total_images) * 1000

fp32_size = get_model_size(model_fp32)
ada_size  = get_model_size(model_adaround)
print(f"FP32 size: {fp32_size:.2f} MB | AdaRound size: {ada_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_ada, img_ms_ada   = benchmark(model_adaround, test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"AdaRound Inference: {batch_ms_ada:.2f} ms/batch | {img_ms_ada:.2f} ms/image")



Total images: 100000, Classes: 200
Train: 85000, Val: 10000, Test: 5000
✅ DataLoaders ready
Missing keys: []
Unexpected keys: []
✅ FP32 GoogLeNet loaded
AdaRounding: conv1.conv.weight
AdaRounding: conv2.conv.weight
AdaRounding: conv3.conv.weight
AdaRounding: inception3a.branch1.conv.weight
AdaRounding: inception3a.branch2.0.conv.weight
AdaRounding: inception3a.branch2.1.conv.weight
AdaRounding: inception3a.branch3.0.conv.weight
AdaRounding: inception3a.branch3.1.conv.weight
AdaRounding: inception3a.branch4.1.conv.weight
AdaRounding: inception3b.branch1.conv.weight
AdaRounding: inception3b.branch2.0.conv.weight
AdaRounding: inception3b.branch2.1.conv.weight
AdaRounding: inception3b.branch3.0.conv.weight
AdaRounding: inception3b.branch3.1.conv.weight
AdaRounding: inception3b.branch4.1.conv.weight
AdaRounding: inception4a.branch1.conv.weight
AdaRounding: inception4a.branch2.0.conv.weight
AdaRounding: inception4a.branch2.1.conv.weight
AdaRounding: inception4a.branch3.0.conv.weight
AdaRound